In [1]:
from pathlib import Path
from utils.model import model_gemini_retry
import logfire
from pydantic_ai import Agent
from dotenv import load_dotenv
from pydantic_ai_harness.subagents import SubAgent, SubAgents
from pydantic_ai_harness.planning import Planning
from pydantic_ai.common_tools.tavily import tavily_search_tool
load_dotenv()

api_key = "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"

PROMPTS_DIR = Path("prompts")


def load_instructions(name: str) -> str:
    """Read an agent's instructions from prompts/<name>.j2."""
    return (PROMPTS_DIR / f"{name}.j2").read_text()

In [2]:
logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: https://logfire-eu.pydantic.dev/johanidler/rain

In [3]:
create_booking_agent = Agent(
    model=model_gemini_retry,
    name="create_booking_agent",
    description="Provide this Agent with exact information, it will then navigate the webstie to create the booking",
    instructions=load_instructions("create_booking_agent"),
    # this one needs a way to access websites and pay for them
)

reserach_hotels_agent = Agent(
    model=model_gemini_retry,
    name="reserach_hotels_agent",
    description="Researches hotel options for a destination and reports back a short list, without booking",
    instructions=load_instructions("reserach_hotels_agent"),
    tools=[tavily_search_tool(api_key)]
)

research_flights_agent = Agent(
    model=model_gemini_retry,
    name="research_flights_agent",
    description="Researches flight options for a route and dates and reports back a short list, without booking",
    instructions=load_instructions("research_flights_agent"),
    tools=[tavily_search_tool(api_key)]
)

In [4]:
travel_agent = Agent(
    model=model_gemini_retry,
    instructions=load_instructions("travel_agent"),
    capabilities=[
        Planning(),
        SubAgents(agents=[
            SubAgent(create_booking_agent),
            SubAgent(reserach_hotels_agent),
            SubAgent(research_flights_agent),
        ])
    ],

)

In [5]:
trip_request = Path("data01.json").read_text()
res = await travel_agent.run(trip_request)
print(res.output)

15:44:44.282 travel_agent run
15:44:44.316   chat gemini-3.6-flash


Traceback (most recent call last):
  File "/Users/johanidler/GitHub/Rain_Hacks/agents/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3773, in run_code
    await eval(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/x9/kzr8hktd3kx0__b9xr1g50cm0000gn/T/ipykernel_64079/3858471611.py", line 2, in <module>
    res = await travel_agent.run(trip_request)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/johanidler/GitHub/Rain_Hacks/agents/.venv/lib/python3.13/site-packages/pydantic_ai/agent/abstract.py", line 647, in run
    node = await agent_run.next(node)  # pyright: ignore[reportArgumentType]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/johanidler/GitHub/Rain_Hacks/agents/.venv/lib/python3.13/site-packages/pydantic_ai/run.py", line 470, in next
    return await self._run_node_with_hooks(node, self._stream_and_advance)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/johanidler/GitHub/